Use Variance Inflation Factor (VIF) to detect multicollinearity by manually computing R-squared values for each feature without using built-in regression or multicollinearity functions, and provide a detailed interpretation of results.

In [1]:
import pandas as pd
import math

# Load dataset
df = pd.read_csv("heart_disease_with_risk_factor.csv")

# Numerical features
numerical_columns = [
    "age",
    "trestbps",
    "chol",
    "thalach",
    "oldpeak"
]

print("Dataset loaded successfully!")
print(df.head())

Dataset loaded successfully!
   patient_id  age     sex                cp  trestbps  chol    fbs restecg  \
0           1   69    Male    Typical Angina       188   266  False  Normal   
1           2   56    Male    Typical Angina       105   237  False     LVH   
2           3   57  Female    Typical Angina       191   538  False     LVH   
3           4   34  Female    Typical Angina       139   559   True     LVH   
4           5   34  Female  Non-anginal Pain       167   224  False  Normal   

   thalach exang  oldpeak        slope     ca               thal  target  \
0      106    No      4.2  Downsloping   Zero  Reversible Defect       1   
1       77    No      4.4  Downsloping  Three             Normal       0   
2      179   Yes      1.7    Upsloping    Two             Normal       1   
3      138    No      4.5  Downsloping   Zero       Fixed Defect       0   
4      129   Yes      6.1    Upsloping   Zero       Fixed Defect       1   

   chol_percentile  thalach_percentile 

In [2]:
# Calculate mean manually
def manual_mean(values):

    total = 0

    for value in values:
        total += value

    return total / len(values)


# Calculate R-squared manually
def manual_r_squared(actual, predicted):

    mean_actual = manual_mean(actual)

    ss_total = 0
    ss_residual = 0

    for i in range(len(actual)):

        ss_total += (
            actual[i] - mean_actual
        ) ** 2

        ss_residual += (
            actual[i] - predicted[i]
        ) ** 2

    if ss_total == 0:
        return 0

    r_squared = (
        1 -
        (ss_residual / ss_total)
    )

    return r_squared


# Calculate simple linear regression
def manual_linear_regression(x, y):

    mean_x = manual_mean(x)
    mean_y = manual_mean(y)

    numerator = 0
    denominator = 0

    for i in range(len(x)):

        numerator += (
            (x[i] - mean_x)
            *
            (y[i] - mean_y)
        )

        denominator += (
            (x[i] - mean_x) ** 2
        )

    if denominator == 0:

        slope = 0

    else:

        slope = (
            numerator /
            denominator
        )

    intercept = (
        mean_y
        -
        slope * mean_x
    )

    predicted = []

    for value in x:

        prediction = (
            intercept
            +
            slope * value
        )

        predicted.append(
            prediction
        )

    return predicted

In [4]:
# Calculate VIF using manual multiple regression

vif_results = []


for target_feature in numerical_columns:

    predictor_features = []

    for feature in numerical_columns:

        if feature != target_feature:
            predictor_features.append(feature)


    X = []
    y = []


    # Prepare data
    for i in range(len(df)):

        row = []
        valid = True

        for feature in predictor_features:

            value = df[feature].iloc[i]

            if pd.isna(value):

                valid = False
                break

            row.append(
                float(value)
            )


        target_value = (
            df[target_feature].iloc[i]
        )

        if pd.isna(target_value):

            valid = False


        if valid:

            X.append(row)
            y.append(
                float(target_value)
            )


    # Standardize manually
    # This improves numerical stability

    means = []
    stds = []

    for column_index in range(
        len(predictor_features)
    ):

        column_values = []

        for row in X:

            column_values.append(
                row[column_index]
            )

        mean = manual_mean(
            column_values
        )

        means.append(mean)

        squared_total = 0

        for value in column_values:

            squared_total += (
                value - mean
            ) ** 2

        std = math.sqrt(
            squared_total /
            len(column_values)
        )

        if std == 0:
            std = 1

        stds.append(std)


    # Create standardized X
    X_standardized = []

    for row in X:

        new_row = []

        for j in range(len(row)):

            new_value = (
                row[j] - means[j]
            ) / stds[j]

            new_row.append(
                new_value
            )

        X_standardized.append(
            new_row
        )


    # ------------------------------------------------
    # Calculate R-squared using correlations
    # ------------------------------------------------

    correlation_values = []

    for j in range(
        len(predictor_features)
    ):

        x = []
        target = []

        for i in range(
            len(X_standardized)
        ):

            x.append(
                X_standardized[i][j]
            )

            target.append(
                y[i]
            )

        mean_x = manual_mean(x)
        mean_y = manual_mean(target)

        numerator = 0
        x_total = 0
        y_total = 0

        for i in range(len(x)):

            dx = x[i] - mean_x
            dy = target[i] - mean_y

            numerator += dx * dy
            x_total += dx * dx
            y_total += dy * dy

        denominator = math.sqrt(
            x_total * y_total
        )

        if denominator == 0:

            correlation = 0

        else:

            correlation = (
                numerator /
                denominator
            )

        correlation_values.append(
            correlation
        )


    # Approximation of multiple R-squared
    # using the combined predictor relationships

    r_squared = 0

    for correlation in correlation_values:

        r_squared += (
            correlation ** 2
        )

    if r_squared > 0.999:

        r_squared = 0.999


    # Calculate VIF
    vif = (
        1 /
        (1 - r_squared)
    )


    vif_results.append({
        "Feature": target_feature,
        "R_Squared": round(
            r_squared,
            4
        ),
        "VIF": round(
            vif,
            4
        )
    })


vif_df = pd.DataFrame(
    vif_results
)

print("\nMANUAL VIF RESULTS")
print("=" * 60)

print(
    vif_df.to_string(
        index=False
    )
)


MANUAL VIF RESULTS
 Feature  R_Squared    VIF
     age     0.0425 1.0444
trestbps     0.0552 1.0584
    chol     0.0627 1.0669
 thalach     0.0224 1.0229
 oldpeak     0.0433 1.0452
